# 03 最速下降法、共轭梯度法与预处理

本节从对称正定矩阵开始。若 $A$ 是对称正定矩阵，求解

$$
Ax=b
$$

等价于最小化二次函数

$$
\phi(x)=\frac12 x^TAx-b^Tx.
$$

梯度为

$$
\nabla \phi(x)=Ax-b=-r,
$$

其中 $r=b-Ax$ 是残差。因此残差既是线性系统误差的度量，也给出了二次型下降方向。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    conjugate_gradient,
    jacobi_preconditioner,
    preconditioned_conjugate_gradient,
    relative_residual,
    steepest_descent,
)


## 最速下降法

最速下降法沿残差方向更新：

$$
x_{k+1}=x_k+\alpha_k r_k.
$$

选择 $\alpha_k$ 使 $\phi(x_k+\alpha r_k)$ 最小。对一元二次函数求导，得到

$$
\alpha_k=\frac{r_k^Tr_k}{r_k^T A r_k}.
$$

这个方法每步都让当前方向最优，但不同方向之间不保持共轭，条件数大时常出现锯齿式缓慢收敛。


In [ ]:
def teaching_steepest_descent(A, b, x0=None, tolerance=1e-8, max_iterations=200):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.asarray(x0, dtype=float).copy()
    r = b - A @ x
    residuals = [np.linalg.norm(r) / np.linalg.norm(b)]
    for k in range(1, max_iterations + 1):
        alpha = (r @ r) / (r @ A @ r)
        x = x + alpha * r
        r = b - A @ x
        residuals.append(np.linalg.norm(r) / np.linalg.norm(b))
        if residuals[-1] <= tolerance:
            break
    return x, np.array(residuals)


## 共轭梯度法

CG 不再每次只沿负梯度方向，而是构造 $A$-共轭方向

$$
p_i^T A p_j=0,\qquad i\ne j.
$$

在精确算术中，$n$ 维 SPD 系统最多 $n$ 步收敛。标准递推为

$$
\alpha_k=\frac{r_k^T r_k}{p_k^T A p_k},
$$

$$
x_{k+1}=x_k+\alpha_k p_k,\qquad r_{k+1}=r_k-\alpha_k A p_k,
$$

$$
\beta_k=\frac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k},\qquad p_{k+1}=r_{k+1}+\beta_k p_k.
$$

实现中不应显式求逆，只需要矩阵向量乘法和向量内积。


In [ ]:
def teaching_cg(A, b, x0=None, tolerance=1e-8, max_iterations=None):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if max_iterations is None:
        max_iterations = len(b)
    x = np.zeros_like(b) if x0 is None else np.asarray(x0, dtype=float).copy()
    r = b - A @ x
    p = r.copy()
    rr = r @ r
    residuals = [np.linalg.norm(r) / np.linalg.norm(b)]
    for k in range(1, max_iterations + 1):
        Ap = A @ p
        alpha = rr / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        residuals.append(np.linalg.norm(r) / np.linalg.norm(b))
        if residuals[-1] <= tolerance:
            break
        rr_new = r @ r
        beta = rr_new / rr
        p = r + beta * p
        rr = rr_new
    return x, np.array(residuals)


## 与正式实现对照

下面用一个小 SPD 系统比较最速下降、CG 和正式实现。CG 通常显著少于最速下降迭代次数。


In [ ]:
A = np.array([
    [6.0, 2.0, 0.0],
    [2.0, 5.0, 1.0],
    [0.0, 1.0, 4.0],
])
b = np.array([1.0, 2.0, 3.0])
exact = np.linalg.solve(A, b)

x_sd, res_sd = teaching_steepest_descent(A, b, tolerance=1e-12, max_iterations=100)
x_cg, res_cg = teaching_cg(A, b, tolerance=1e-12)
formal_sd = steepest_descent(A, b, tolerance=1e-12, max_iterations=100)
formal_cg = conjugate_gradient(A, b, tolerance=1e-12)

print("最速下降误差：", np.linalg.norm(formal_sd.value - exact), "迭代次数：", formal_sd.iterations)
print("CG 误差：", np.linalg.norm(formal_cg.value - exact), "迭代次数：", formal_cg.iterations)
print("教学/正式 CG 差异：", np.linalg.norm(x_cg - formal_cg.value))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(formal_sd.residual_norms, "o-", label="最速下降")
ax.semilogy(formal_cg.residual_norms, "s-", label="CG")
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("最速下降与 CG 残差历史")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 条件数对收敛速度的影响

SPD 矩阵的条件数越大，等高线越狭长，最速下降越容易锯齿前进。CG 也会受到条件数影响，但通常明显更稳健。


In [ ]:
def diagonal_spd(condition_number):
    values = np.linspace(1.0, condition_number, 8)
    return np.diag(values)

for kappa in [5, 50, 500]:
    A_k = diagonal_spd(kappa)
    b_k = np.ones(8)
    sd = steepest_descent(A_k, b_k, tolerance=1e-8, max_iterations=1000)
    cg = conjugate_gradient(A_k, b_k, tolerance=1e-8, max_iterations=8)
    print(f"cond={kappa:4d}  SD iters={sd.iterations:4d}  CG iters={cg.iterations:2d}")


## 预处理共轭梯度法

预处理希望用一个容易求解的矩阵 $M$ 近似 $A$，把问题变成条件数更好的形式。PCG 的核心是每步求

$$
Mz_k=r_k.
$$

Jacobi 预处理取 $M=\operatorname{diag}(A)$，只需要把残差除以对角元。它不总是显著改善，但容易实现，是理解预处理的第一步。


In [ ]:
A_bad = np.diag([1.0, 10.0, 100.0, 1000.0]) + 0.05 * np.ones((4, 4))
A_bad = 0.5 * (A_bad + A_bad.T)
b_bad = np.ones(4)
preconditioner = jacobi_preconditioner(A_bad)
plain = conjugate_gradient(A_bad, b_bad, tolerance=1e-10, max_iterations=20)
pcg = preconditioned_conjugate_gradient(A_bad, b_bad, preconditioner=preconditioner, tolerance=1e-10, max_iterations=20)
print("CG 迭代次数：", plain.iterations, "最终残差：", plain.residual_norms[-1])
print("PCG 迭代次数：", pcg.iterations, "最终残差：", pcg.residual_norms[-1])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(plain.residual_norms, "o-", label="CG")
ax.semilogy(pcg.residual_norms, "s-", label="Jacobi PCG")
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("预处理前后的残差历史")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 适用条件和失败情形

CG 和 PCG 要求矩阵为对称正定。若矩阵非对称、不定或严重病态，标准 CG 可能出现非正曲率方向或停滞。实际大规模问题中，通常不会显式构造逆矩阵，而是只提供矩阵向量乘法和预处理求解器。本章的实现面向教学和小规模验证，后续 Poisson 稀疏系统会继续使用同一思想。


## 本节小结

最速下降法从二次型最小化出发，每步沿残差方向做一维精确线搜索；CG 通过构造 $A$-共轭方向，避免重复搜索已经处理过的方向，在 SPD 系统上通常远快于最速下降。PCG 的关键是通过预处理改善条件性，但预处理器本身也有计算成本。选择迭代法时，应同时看矩阵结构、条件数、每步成本和可用预处理。
